### Librerias

In [ ]:
import pandas as pd

### Carga de datos

In [ ]:
df = pd.read_csv('../../data/raw_data_16-03-2026.csv',index_col=0)
df.info()

In [ ]:
#Total de registros
print("Total filas:", len(df))
#Total de IDs únicos
print("IDs únicos:", df['apartment_id'].nunique())
#Total de registros duplicados 
print("Duplicados (estilo SQL):", len(df) - df['apartment_id'].nunique())

df['apartment_id'].isna().sum()

### Limpieza


Nulos

In [ ]:
df.isnull().sum()

In [ ]:
df[df['name'].isnull()]
df['description'] = df['description'].fillna('(sin contestar)')
df['neighbourhood_district'] = df['neighbourhood_district'].fillna('(sin contestar)')
df['amenities_list'] = df['amenities_list'].fillna('(sin contestar)')

#### Situación con nulos en precio
- 2.14% de nulos --> bajo volumen
- No son aleatorios --> sesgo geográfico fuerte (Mallorca, Menorca)
- 161 apartamentos sin precio --> no recuperables
- 5 casos inconsistentes (registros duplicados con y sin precio) --> resueltos priorizando la observación con precio

In [ ]:
total = len(df)
nulls = df['price'].isna().sum()
pct_nulls = nulls / total * 100

print(f"Total: {total}")
print(f"Nulls: {nulls}")
print(f"% Nulls: {pct_nulls:.2f}")

In [ ]:
agg = (
    df.groupby('apartment_id')
      .agg(
          total_rows=('price', 'size'),
          with_price=('price', lambda x: x.notna().sum())
      )
      .reset_index()
)

alojamientos_solo_null = agg[(agg['with_price'] == 0)]
alojamientos_mixtos = agg[(agg['with_price'] > 0) & (agg['with_price'] < agg['total_rows'])]
alojamientos_completos = agg[(agg['with_price'] == agg['total_rows'])]

print(len(alojamientos_solo_null), len(alojamientos_mixtos), len(alojamientos_completos))

- total_distribucion: % de anuncios por ciudad sobre el total del dataset  
- null_distribucion: % de anuncios con precio nulo dentro del total de nulos  
- null_count: número absoluto de anuncios con precio nulo  
- ratio: relación entre null_distribucion y total_distribucion (indica si una ciudad está sobrerrepresentada (>1) o infrarepresentada (<1) en los nulos)

In [ ]:
df['price_missing'] = df['price'].isna()

dist_null = (df[df['price_missing']].city.value_counts(normalize=True)*100).round(2)
dist_total = (df.city.value_counts(normalize=True)*100).round(2)
count_city = df['city'].value_counts()
count_null_city = df[df['price'].isna()]['city'].value_counts()

comparado = pd.concat([count_city,dist_total,count_null_city, dist_null], axis=1)
comparado.columns = ['cantidad_total','total_distribucion', 'null_count', 'null_distribucion']
comparado['ratio'] = (comparado['null_distribucion'] / comparado['total_distribucion']).round(2) 
comparado

- Mallorca presenta una fuerte sobrerrepresentación de nulos:
  - 16.5% del total de anuncios
  - 45.6% de los nulos
  - ratio: 2.77  
  - casi 3 veces más nulos de lo esperado

- Menorca también muestra sobrerrepresentación:
  - ratio: 1.64

- El resto de ciudades están infrarepresentadas:
  - ratios entre 0.1 y 0.75

- Eliminar los registros con precio nulo supone un sesgo geográfico --> reduce artificialmente el peso de Mallorca y Menorca
- Mantenerlos sin gestionar puede afectar a análisis que dependan indirectamente del precio

In [ ]:
# df_null = df[df['price'].isna()].copy()           # Si queremos quedarnos solo con los registros con precio null, creamos este nuevo dataframe
# df_clean = df[df['price'].notna()].copy()         # Si queremos quedarnos solo con los registros con precio, creamos este nuevo dataframe
df['price_missing'] = df['price'].isna()            # Si nos quedamos con el dataseet completo, añadimos esta columna para marcar los registros con precio faltante.

Duplicados

Se ordena el dataset por identificador y fecha para conservar únicamente el registro más reciente en la tabla principal, archivando las versiones anteriores en un DataFrame secundario a modo de histórico.


In [ ]:
df = df.sort_values(by=['apartment_id','insert_date'], ascending=[True, False])


dfanunciosantiguos = df[df.duplicated(subset=['apartment_id'],keep=False)].copy()

df = df.drop_duplicates(subset=['apartment_id'], keep='first')

In [ ]:
print("Total filas:", len(df))
print("IDs únicos:", df['apartment_id'].nunique())
print("Duplicados (estilo SQL):", len(df) - df['apartment_id'].nunique())
df['apartment_id'].isna().sum()

In [ ]:
es_unico = df['apartment_id'].is_unique
print(f"¿Son todos los IDs únicos?: {es_unico}")

In [ ]:
# Estandarización de texto: formato título para los nombres de las ciudades
df['city'] = df['city'].str.title()

### Transformación 

Para garantizar la integridad del análisis, se ha llevado a cabo un proceso de estandarización estructural del dataset. Esto ha incluido el casting de variables (conversión de las columnas a sus tipos de datos nativos correspondientes, como numéricos, booleanos o fechas) para permitir operaciones matemáticas correctas. Asimismo, se han corregido inconsistencias y anomalías de formato detectadas en varias columnas, asegurando que la información sea coherente y esté optimizada para la fase de modelado.

Se recalcula la columna de reviews_per_month ya que el equipo se ha dado cuenta de que los numeros no son correctos
Convierto las columnas de las reseñas a formato fecha para poder calcular el tiempo pasado desde 'first_review_date'(primera fecha de la que disponemos) hasta la 'insert_date'(ultima fecha de la que disponemos) para que el calculo sea mas cercano a la realidad

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7693 entries, 0 to 7999
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7693 non-null   int64         
 1   name                         7690 non-null   object        
 2   description                  7693 non-null   object        
 3   host_id                      7693 non-null   int64         
 4   neighbourhood_name           7693 non-null   object        
 5   neighbourhood_district       7693 non-null   object        
 6   room_type                    7693 non-null   object        
 7   accommodates                 7693 non-null   int64         
 8   bathrooms                    7650 non-null   Int64         
 9   bedrooms                     7655 non-null   Int64         
 10  beds                         7685 non-null   Int64         
 11  amenities_list               7693 non-null   obj

In [ ]:
df['bathrooms']=df['bathrooms'].astype('Int64')

df['beds']=df['beds'].astype('Int64')

df['bedrooms']=df['bedrooms'].astype('Int64')



In [ ]:



df['first_review_date']= pd.to_datetime(df['first_review_date'], dayfirst=True)

df['last_review_date']= pd.to_datetime(df['last_review_date'], dayfirst=True)
df['insert_date'] = pd.to_datetime(df['insert_date'], dayfirst=True)

# Calculamos los meses transcurridos
meses = (df['insert_date'] - df['first_review_date']).dt.days / 30.4



#df['reviews_per_month'] = round(((df['number_of_reviews'] / meses)),2)



In [ ]:


df['review_scores_rating']=df['review_scores_rating']/10
df['review_scores_rating']=df['review_scores_rating'].astype('Int64')

# Esta línea convierte lo que no sea 'VERDADERO' a False (Asumiendo que si esta nulo es porque no tiene disponibilidad)
df['has_availability'] = df['has_availability'] == 'VERDADERO'

df['is_instant_bookable'] = df['is_instant_bookable'] == 'VERDADERO'



In [ ]:
# Resumen de rangos para todas las columnas de números
resumen_limpieza = df.select_dtypes(include='number').agg(['min', 'max'])
print(resumen_limpieza)

### Creación de variables

Se ha generado una nueva variable booleana para identificar los apartamentos con valoraciones superiores a 80

In [ ]:
df['reviews 80+']=df['review_scores_rating'] >= 80 

In [ ]:
df.info()

In [ ]:
df

Se genera una archivo CSV con los datos limpios y otro con los anuncios antiguos

In [ ]:
# Generación del CSV bloqueada (el archivo ya se encuentra en el directorio del proyecto).
#df.to_csv('../../data/clean_data_16-03-2026.csv', index=False)

#dfanunciosantiguos.to_csv('../../data/Anuncios antiguos_09-03-2026.csv', index=False)